In [1]:
%pip install openai pydub python-dotenv audioop-lts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [audioop-lts]
Note: you may need to restart the kernel to use updated packages.


In [2]:
%conda install -c conda-forge ffmpeg -y

2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3/envs/ironhack

  added / updated specs:
    - ffmpeg


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    aom-3.9.1                  |       h7bae524_0         2.1 MB  conda-forge
    ca-certificates-2026.10.23 |       hbd8a1cb_0         127 KB  conda-forge
    cairo-1.18.4               |       h6a3b0d2_0         875 KB  conda-forge
    dav1d-1.2.1                |       hb547adb_0         309 KB  conda-forge
    ffmpeg-7.1.0               | gpl_h92f8dbd_105         8.7 MB  conda-forge
    font-ttf-dejavu-sans-mono-2.37|       hab24e00_0         388 KB  conda-forge
    font-ttf-inconsolata-3.000 |       h77eed37_0          94 KB  conda-forge
    font-ttf-source-code-pro-2.038|       h77eed

In [2]:
# Step 1: Setting up the environment
import os, sys, shutil
from dotenv import load_dotenv
from openai import OpenAI
from pydub import AudioSegment

# Load the API key from .env (never hardcoded, .env is excluded via .gitignore)
# .strip() removes stray spaces/line breaks that would break the HTTP header
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:
    raise ValueError("OPENAI_API_KEY not found")

client = OpenAI(api_key=api_key)

# Create the project folders (exist_ok avoids errors on re-runs)
# audio/ = input recordings, chunks/ = split audio, transcripts/ = exports
for d in ["audio", "chunks", "transcripts"]:
    os.makedirs(d, exist_ok=True)

# Checkpoint: verify Python version, ffmpeg installation and Whisper access
print("Python:", sys.version.split()[0])
print("ffmpeg:", shutil.which("ffmpeg"))
print("Whisper verfügbar:", any(m.id == "whisper-1" for m in client.models.list()))

Python: 3.14.7
ffmpeg: /opt/anaconda3/envs/ironhack/bin/ffmpeg
Whisper verfügbar: True


In [3]:
# Step 2: Load the sample meeting audio and check its properties
from pathlib import Path

# Find all supported audio files in the audio/ folder
audio_files = [f for f in Path("audio").iterdir()
               if f.suffix.lower() in [".mp3", ".wav", ".m4a", ".mp4", ".webm"]]
print("Files found:", [f.name for f in audio_files])

# Use the first file found and load it with pydub
AUDIO_PATH = audio_files[0]
audio = AudioSegment.from_file(AUDIO_PATH)

# Check properties relevant for the Whisper API:
# - file size: the API rejects files larger than 25 MB -> chunking needed
# - duration: determines how many chunks we'll create in Step 6
size_mb = AUDIO_PATH.stat().st_size / (1024 * 1024)
print(f"\nFile:         {AUDIO_PATH.name}")
print(f"File size:    {size_mb:.2f} MB  (Whisper limit: 25 MB)")
print(f"Duration:     {len(audio) / 1000 / 60:.1f} minutes")  # pydub length is in ms
print(f"Sample rate:  {audio.frame_rate} Hz")
print(f"Channels:     {audio.channels}")

Files found: ['CA138clip.mp3']

File:         CA138clip.mp3
File size:    0.89 MB  (Whisper limit: 25 MB)
Duration:     1.4 minutes
Sample rate:  44100 Hz
Channels:     1


In [4]:
# Step 3: Basic transcription (without chunking)
import time

# Cut out the first 30 seconds (pydub works in milliseconds)
SNIPPET_SECONDS = 30
snippet = audio[:SNIPPET_SECONDS * 1000]

# Save the snippet as its own file, since the API expects a file upload
# Stored in chunks/ (excluded via .gitignore), not in audio/,
# so Step 2 doesn't pick it up as the "first audio file" on re-runs
snippet_path = Path("chunks") / "snippet_first_30s.mp3"
snippet.export(snippet_path, format="mp3")

# Send the snippet to Whisper
# The file is opened fresh with "with open(...)", so it is closed again afterwards
start_time = time.time()
with open(snippet_path, "rb") as f:
    response = client.audio.transcriptions.create(
        model="whisper-1",
        file=f
    )
elapsed = time.time() - start_time

# Inspect the response: the default format only returns the plain text,
# no timestamps or language (those come with "verbose_json" later)
print("Response type:", type(response).__name__)
print(f"API call took: {elapsed:.1f} s\n")
print("Transcription:")
print("-" * 40)
print(response.text)

Response type: Transcription
API call took: 4.1 s

Transcription:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what do they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to New York. Oh yes, I'll be so glad when they land back now. I think that's pretty well a fact because they've landed so many safely now.


In [5]:
# Play the snippet to manually verify the transcription
from IPython.display import Audio, display
display(Audio(str(snippet_path)))

In [6]:
# Step 4: Transcription with prompts (guided approach)
import json

# Context prompt: Whisper does NOT follow instructions here.
# It treats the prompt as text "spoken before" the audio and adopts
# its vocabulary, spelling and style (only the last 224 tokens are used).
# The prompt describes the topic we identified in Step 3: the Apollo moon landing.
CONTEXT_PROMPT = (
    "A casual conversation about the Apollo moon landing. "
    "Topics: NASA, astronauts, lunar surface, kangaroo hop, splashdown, "
    "ticker-tape parade in New York."
)

# temperature=0 makes the output as deterministic as possible,
# so differences between prompted and unprompted runs come from
# the prompt and not from random sampling
with open(snippet_path, "rb") as f:
    prompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        prompt=CONTEXT_PROMPT,
        temperature=0
    )

print("Prompt used:")
print(CONTEXT_PROMPT)
print("\nTranscription WITH prompt:")
print("-" * 40)
print(prompted.text)

# Save the result so it can be compared in Step 5 and referenced in lab_proof.md
with open("transcripts/step4_prompted.json", "w", encoding="utf-8") as f:
    json.dump({"prompt": CONTEXT_PROMPT, "text": prompted.text}, f, indent=2, ensure_ascii=False)
print("\nSaved to transcripts/step4_prompted.json")

Prompt used:
A casual conversation about the Apollo moon landing. Topics: NASA, astronauts, lunar surface, kangaroo hop, splashdown, ticker-tape parade in New York.

Transcription WITH prompt:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Yes, I'll be so glad when they land back now, but I think that's pretty well a fact because they've landed so many safely now.

Saved to transcripts/step4_prompted.json


In [7]:
# Step 5: Transcription without prompts (unguided approach)
import difflib

# Same snippet, same settings (temperature=0) as Step 4 - only the prompt is missing.
# This makes the comparison fair: any difference is caused by the prompt.
with open(snippet_path, "rb") as f:
    unprompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        temperature=0
    )

print("Transcription WITHOUT prompt:")
print("-" * 40)
print(unprompted.text)

# Save the result next to the prompted version
with open("transcripts/step5_unprompted.json", "w", encoding="utf-8") as f:
    json.dump({"prompt": None, "text": unprompted.text}, f, indent=2, ensure_ascii=False)

# Word-level comparison of both versions
# difflib finds the differences between the two word lists:
# [-word-] = only in the unprompted version, {+word+} = only in the prompted version
words_without = unprompted.text.split()
words_with = prompted.text.split()
matcher = difflib.SequenceMatcher(None, words_without, words_with)

print("\nDifferences (without -> with prompt):")
print("-" * 40)
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag != "equal":
        removed = " ".join(words_without[i1:i2])
        added = " ".join(words_with[j1:j2])
        print(f"{tag:8} [-{removed}-]  {{+{added}+}}")

# Similarity score: 1.0 = identical, 0.0 = completely different
print(f"\nWord-level similarity: {matcher.ratio():.1%}")

Transcription WITHOUT prompt:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what do they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to New York. Oh yes, I'll be so glad when they land back now. I think that's pretty well a fact because they've landed so many safely now.

Differences (without -> with prompt):
----------------------------------------
delete   [-do-]  {++}
replace  [-New York. Oh yes,-]  {+Earth. Yes,+}
replace  [-now.-]  {+now, but+}

Word-level similarity: 93.3%


In [8]:
# Step 6: Implementing audio chunking
# Split the full recording into fixed-length chunks.
# The sample is only 1.4 min (0.89 MB), far below the 25 MB API limit,
# so 30-second chunks are used to demonstrate the technique.
# For real 2h+ meetings, 10-20 minute chunks would be more sensible.
CHUNK_SECONDS = 30
chunk_ms = CHUNK_SECONDS * 1000

chunks_info = []  # stores file path and start/end time of each chunk

# Step through the audio in 30-second steps: 0 ms, 30000 ms, 60000 ms, ...
for i, start_ms in enumerate(range(0, len(audio), chunk_ms)):
    end_ms = min(start_ms + chunk_ms, len(audio))  # last chunk may be shorter
    chunk = audio[start_ms:end_ms]

    chunk_path = Path("chunks") / f"chunk_{i:02d}.mp3"  # chunk_00, chunk_01, ...
    chunk.export(chunk_path, format="mp3")

    # The start offset is essential for Step 7: Whisper's timestamps
    # restart at 0 for every chunk, so we must add this offset back
    chunks_info.append({
        "index": i,
        "path": str(chunk_path),
        "start_sec": start_ms / 1000,
        "end_sec": end_ms / 1000,
    })

# Checkpoint: list all chunks with their position in the original recording
print(f"Created {len(chunks_info)} chunks of max. {CHUNK_SECONDS} s:\n")
for c in chunks_info:
    print(f"  {c['path']:22} {c['start_sec']:6.1f}s - {c['end_sec']:6.1f}s")

Created 3 chunks of max. 30 s:

  chunks/chunk_00.mp3       0.0s -   30.0s
  chunks/chunk_01.mp3      30.0s -   60.0s
  chunks/chunk_02.mp3      60.0s -   86.2s


In [9]:
# Step 7: Transcribing chunks with timestamps
all_segments = []  # combined segments of the whole recording

for c in chunks_info:
    print(f"Transcribing {c['path']} (offset {c['start_sec']:.0f}s)...")
    try:
        with open(c["path"], "rb") as f:
            result = client.audio.transcriptions.create(
                model="whisper-1",
                file=f,
                response_format="verbose_json",       # needed for timestamps
                timestamp_granularities=["segment"],  # sentence-level timestamps
                prompt=CONTEXT_PROMPT,                # guided approach (more accurate in Step 4/5)
                temperature=0
            )
    except Exception as e:
        # Error handling: one failed chunk should not stop the whole transcription
        print(f"  Error in chunk {c['index']}: {e}")
        continue

    # Whisper's timestamps restart at 0 for every chunk.
    # Adding the chunk's start offset maps them back to the original recording.
    for seg in result.segments:
        all_segments.append({
            "start": round(seg.start + c["start_sec"], 2),
            "end": round(seg.end + c["start_sec"], 2),
            "text": seg.text.strip(),
            "chunk": c["index"],
        })

    # Short pause between calls to avoid hitting API rate limits (Troubleshooting Issue 3)
    time.sleep(1)

# Helper: convert seconds into a readable mm:ss format
def fmt(seconds):
    return f"{int(seconds // 60):02d}:{seconds % 60:05.2f}"

# Checkpoint: print the combined transcript with corrected timestamps
print(f"\n{len(all_segments)} segments in total:\n")
for s in all_segments:
    print(f"[{fmt(s['start'])} - {fmt(s['end'])}] (chunk {s['chunk']})  {s['text']}")

# Sanity check: the last segment should end close to the total duration (86.2 s)
print(f"\nLast segment ends at {all_segments[-1]['end']:.1f}s "
      f"(audio length: {len(audio) / 1000:.1f}s)")

Transcribing chunks/chunk_00.mp3 (offset 0s)...
Transcribing chunks/chunk_01.mp3 (offset 30s)...
Transcribing chunks/chunk_02.mp3 (offset 60s)...

14 segments in total:

[00:00.00 - 00:13.00] (chunk 0)  It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that.
[00:13.00 - 00:15.00] (chunk 0)  Who named it that? I don't know.
[00:15.00 - 00:19.00] (chunk 0)  I bet those men are going to get quite a reception when they get back to Earth.
[00:19.00 - 00:30.00] (chunk 0)  Yes, I'll be so glad when they land back now, but I think that's pretty well a fact because they've landed so many safely now.
[00:31.00 - 00:35.00] (chunk 1)  I feel relieved. Just getting off of the moon was the thing that was...
[00:35.00 - 00:38.00] (chunk 1)  Have they met with the one that was circling?
[00:38.00 - 00:44.00] (chunk 1)  Yes, they've rendezvoused. So I understand. That wasn't shown either, so I...
[00:44.0

In [10]:
# Step 8: Exporting transcriptions with timestamps in multiple formats

# --- 1. Human-readable TXT: one line per segment with [mm:ss] timestamps ---
txt_path = Path("transcripts") / "transcript.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"Transcript of {AUDIO_PATH.name}\n")
    f.write(f"Model: whisper-1 | Chunks: {len(chunks_info)} x {CHUNK_SECONDS}s | Prompt: guided\n\n")
    for s in all_segments:
        f.write(f"[{fmt(s['start'])} - {fmt(s['end'])}]  {s['text']}\n")

# --- 2. SRT (subtitle format) ---
# Each block: running number, "HH:MM:SS,mmm --> HH:MM:SS,mmm", text, empty line.
# Note: SRT uses a comma (not a dot) before the milliseconds.
def srt_time(seconds):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

srt_path = Path("transcripts") / "transcript.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    for i, s in enumerate(all_segments, start=1):  # SRT numbering starts at 1
        f.write(f"{i}\n")
        f.write(f"{srt_time(s['start'])} --> {srt_time(s['end'])}\n")
        f.write(f"{s['text']}\n\n")

# --- 3. JSON: structured data incl. metadata, easy to search or reload ---
json_path = Path("transcripts") / "transcript.json"
export = {
    "source_file": AUDIO_PATH.name,
    "model": "whisper-1",
    "prompt": CONTEXT_PROMPT,
    "chunk_seconds": CHUNK_SECONDS,
    "duration_sec": round(len(audio) / 1000, 2),
    "full_text": " ".join(s["text"] for s in all_segments),
    "segments": all_segments,
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2, ensure_ascii=False)

# Checkpoint: confirm all files exist and preview the first lines of each
for p in [txt_path, srt_path, json_path]:
    print(f"=== {p} ({p.stat().st_size} bytes) ===")
    print("".join(open(p, encoding="utf-8").readlines()[:6]))

=== transcripts/transcript.txt (1493 bytes) ===
Transcript of CA138clip.mp3
Model: whisper-1 | Chunks: 3 x 30s | Prompt: guided

[00:00.00 - 00:13.00]  It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that.
[00:13.00 - 00:15.00]  Who named it that? I don't know.
[00:15.00 - 00:19.00]  I bet those men are going to get quite a reception when they get back to Earth.

=== transcripts/transcript.srt (1557 bytes) ===
1
00:00:00,000 --> 00:00:13,000
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that.

2
00:00:13,000 --> 00:00:15,000

=== transcripts/transcript.json (3716 bytes) ===
{
  "source_file": "CA138clip.mp3",
  "model": "whisper-1",
  "prompt": "A casual conversation about the Apollo moon landing. Topics: NASA, astronauts, lunar surface, kangaroo hop, splashdown, ticker-tape parade in 